In [ ]:
# seal0

In [ ]:
import json
import re

# ========== 配置路径 ==========
input_path  = "/home/shaolingxuan/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1/test.8.10,20:39.json"
output_path = "/home/shaolingxuan/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1/all_cleaned.json"

# ========== 1. 读取已有的 all_extracted.json ==========
try:
    with open(output_path, "r", encoding="utf-8") as f:
        all_data = json.load(f)
    print(f"读取已有数据 {len(all_data)} 条")
except (FileNotFoundError, json.JSONDecodeError):
    all_data = []
    print("输出文件不存在或为空，从头开始")

# ========== 2. 读取输入 JSON 文件 ==========
with open(input_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# 如果是单条 dict，包装成 list
if isinstance(raw_data, dict):
    raw_data = [raw_data]

print(f"输入文件共 {len(raw_data)} 条数据")

# ========== 3. 正则匹配 \boxed{...} ==========
pattern = r'\\boxed\{(.*)\}'    # ← 用非贪婪 .*?，见下方说明。保持和webthinker一致，就用贪婪模式

matched_count   = 0
unmatched_count = 0

for item in raw_data:
    # 兼容 "Output" 和 "output" 两种字段名
    output = item.get("Output", item.get("output"))
    if output is None:
        print(f"  [跳过] id={item.get('id')} 无 Output 字段")
        unmatched_count += 1
        continue

    matches = re.findall(pattern, output)
    if matches:
        pred_answer = matches[-1]  # 取最后一个匹配
        extracted = {
            "id":           item.get("id"),
            "Question":     item.get("Question"),
            "answer":       item.get("answer"),
            "Output":       output,
            "Pred_Answer":  pred_answer
        }
        all_data.append(extracted)
        matched_count += 1
        print(f"  [匹配] id={item.get('id')}  Pred_Answer={pred_answer}")
    else:
        unmatched_count += 1
        print(f"  [未匹配] id={item.get('id')}  Output 中无 \\boxed{{}}")

# ========== 4. 写回 all_extracted.json ==========
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\n完成！匹配成功 {matched_count} 条，未匹配 {unmatched_count} 条")
print(f"输出文件现有 {len(all_data)} 条数据，路径: {output_path}")


输出文件不存在或为空，从头开始
输入文件共 3 条数据
  [未匹配] id=41  Output 中无 \boxed{}
  [匹配] id=54  Pred_Answer=0
  [未匹配] id=60  Output 中无 \boxed{}

完成！匹配成功 1 条，未匹配 2 条
输出文件现有 1 条数据，路径: /home/shaolingxuan/project/Search-o1/outputs/runs.baselines/seal0.qwen3.5-9b.search_o1/all.json


In [ ]:
    {
    "id": 15,
    "Question": "<|im_start|>user\nYou are a reasoning assistant with the ability to perform web searches to help you answer the user's question accurately. You have special tools:\n\n- To perform a search: write <|begin_search_query|> your query here <|end_search_query|>.\nThen, the system will search and analyze relevant web pages, then provide you with helpful information in the format <|begin_search_result|> ...search results... <|end_search_result|>.\n\nYou can repeat the search process multiple times if necessary. The maximum number of search attempts is limited to 10.\n\nOnce you have all the information you need, continue your reasoning.\n\nExample:\nQuestion: \"Alice David is the voice of Lara Croft in a video game developed by which company?\"\nAssistant thinking steps:\n- I need to find out who voices Lara Croft in the video game.\n- Then, I need to determine which company developed that video game.\n\nAssistant:\n<|begin_search_query|>Alice David Lara Croft voice<|end_search_query|>\n\n(System returns processed information from relevant web pages)\n\nAssistant thinks: The search results indicate that Alice David is the voice of Lara Croft in a specific video game. Now, I need to find out which company developed that game.\n\nAssistant:\n<|begin_search_query|>video game developed by Alice David Lara Croft<|end_search_query|>\n\n(System returns processed information from relevant web pages)\n\nAssistant continues reasoning with the new information...\n\nRemember:\n- Use <|begin_search_query|> to request a web search and end with <|end_search_query|>.\n- When done searching, continue your reasoning.\n\nPlease answer the following question. You should think step by step to solve it.\n\nProvide your final answer in the format \\boxed{YOUR_ANSWER}.\n\nQuestion:\nHow many of the top 30 most-liked YouTube videos belong to MrBeast?<|im_end|>\n<|im_start|>assistant\n<think>\n",
    "answer": "9",
    "Output": "The user is asking about the top 30 most-liked YouTube videos and how many of them belong to MrBeast. I need to search for this information.\n\nLet me search for the top most-liked YouTube videos to find out which ones are by MrBeast.\n</think>\n\n<|begin_search_query|>top 30 most-liked YouTube videos MrBeast<|end_search_query|>\n\n<|begin_search_result|>Based on the search results from the Wikipedia list of the top 30 most-liked YouTube videos, there are **7** videos by MrBeast currently appearing in the top 30. The specific videos, their ranking, and approximate like counts are as follows:1.  **#1:** \"Would You Fly to Paris for a Baguette?\" (~57.09 million likes) - *Note: This is a YouTube Short.*2.  **#6:** \"Giving iPhones Instead Of Candy on Halloween\" (~40.16 million likes)3.  **#9:** \"Katana Vs Bullet\" (~37.13 million likes)4.  **#10:** \"The Rock Vs MrBeast For $100,000\" (~35.65 million likes)5.  **#21:** \"Make This Video The Most Liked Video On Youtube\" (~30.33 million likes)6.  **#24:** \"I Sent a Subscriber to Disneyland\" (~29.78 million likes)7.  **#25:** \"Gordon Ramsay Tries Most Expensive Chocolate Bar\" (~29.62 million likes)*Note: The provided search snippet for the Wikipedia list covers ranks 1 through 25. While the list is titled \"Top 30,\" the specific data for ranks 26–30 was not fully included in the retrieved text. However, the 7 videos listed above are confirmed to be in the top 30.*- **Step 5:** Based on the search results, I found that MrBeast has 7 videos in the top 30 most-liked YouTube videos. The top video is \"Would You Fly to Paris for a Baguette?\". I have the necessary information to answer the user's question about the count and list the videos.<|end_search_result|>\n\nAssistant thinks: Based on the search results, I found that MrBeast has 7 videos in the top 30 most-liked YouTube videos. The search results show the following videos by MrBeast in the top 30:\n\n1. #1: \"Would You Fly to Paris for a Baguette?\" (~57.09 million likes)\n2. #6: \"Giving iPhones Instead Of Candy on Halloween\" (~40.16 million likes)\n3. #9: \"Katana Vs Bullet\" (~37.13 million likes)\n4. #10: \"The Rock Vs MrBeast For $100,000\" (~35.65 million likes)\n5. #21: \"Make This Video The Most Liked Video On Youtube\" (~30.33 million likes)\n6. #24: \"I Sent a Subscriber to Disneyland\" (~29.78 million likes)\n7. #25: \"Gordon Ramsay Tries Most Expensive Chocolate Bar\" (~29.62 million likes)\n\nSo the answer is 7 videos.\n\n\\boxed{7}",
    "Pred_Answer": "7"
  },
  {
    "id": 33,
    "Question": "<|im_start|>user\nYou are a reasoning assistant with the ability to perform web searches to help you answer the user's question accurately. You have special tools:\n\n- To perform a search: write <|begin_search_query|> your query here <|end_search_query|>.\nThen, the system will search and analyze relevant web pages, then provide you with helpful information in the format <|begin_search_result|> ...search results... <|end_search_result|>.\n\nYou can repeat the search process multiple times if necessary. The maximum number of search attempts is limited to 10.\n\nOnce you have all the information you need, continue your reasoning.\n\nExample:\nQuestion: \"Alice David is the voice of Lara Croft in a video game developed by which company?\"\nAssistant thinking steps:\n- I need to find out who voices Lara Croft in the video game.\n- Then, I need to determine which company developed that video game.\n\nAssistant:\n<|begin_search_query|>Alice David Lara Croft voice<|end_search_query|>\n\n(System returns processed information from relevant web pages)\n\nAssistant thinks: The search results indicate that Alice David is the voice of Lara Croft in a specific video game. Now, I need to find out which company developed that game.\n\nAssistant:\n<|begin_search_query|>video game developed by Alice David Lara Croft<|end_search_query|>\n\n(System returns processed information from relevant web pages)\n\nAssistant continues reasoning with the new information...\n\nRemember:\n- Use <|begin_search_query|> to request a web search and end with <|end_search_query|>.\n- When done searching, continue your reasoning.\n\nPlease answer the following question. You should think step by step to solve it.\n\nProvide your final answer in the format \\boxed{YOUR_ANSWER}.\n\nQuestion:\nAmong women’s soccer players with 100 or more international goals, who has the highest goals-per-match ratio?<|im_end|>\n<|im_start|>assistant\n<think>\n",
    "answer": "Julie Fleeting",
    "Output": "The user is asking about women's soccer players with 100 or more international goals and wants to know who has the highest goals-per-match ratio among them.\n\nI need to search for information about women's soccer players with 100+ goals and their goals-per-match ratios.\n\nLet me start by searching for women's soccer players with 100+ international goals.\n</think>\n\n<|begin_search_query|>women's soccer players 100+ international goals list<|end_search_query|>\n\n<|begin_search_result|>Based on the searched web pages, I found comprehensive information about women's soccer players with 100+ international goals. Here is the key data:**Complete List of Women's Soccer Players with 100+ International Goals (as of March 2026):**| Rank | Player | Nation | Goals | Caps | Goals per Match ||------|--------|--------|-------|------|-----------------|| 1 | Christine Sinclair | Canada | 190 | 331 | 0.57 || 2 | Abby Wambach | United States | 184 | 255 | 0.72 || 3 | Mia Hamm | United States | 158 | 276 | 0.57 || 4 | Maysa Jbarah | Jordan | 134 | 140 | 0.96 || 5 | Carli Lloyd | United States | 134 | 316 | 0.42 || 6 | Kristine Lilly | United States | 130 | 354 | 0.37 || 7 | Birgit Prinz | Germany | 128 | 214 | 0.60 || 8 | Alex Morgan | United States | 123 | 224 | 0.55 || 9 | Marta | Brazil | 121 | 200 | 0.61 || 10 | Julie Fleeting | Scotland | 116 | 118 | 0.98 || 11 | Patrizia Panico | Italy | 110 | 204 | 0.54 || 12 | Elisabetta Vignotto | Italy | 107 | 109 | 0.98 || 13 | Michelle Akers | United States | 107 | 155 | 0.69 || 14 | Carolina Morace | Italy | 105 | 153 | 0.69 || 15 | Vivianne Miedema | Netherlands | 104 | 132 | 0.79 || 16 | Portia Modise | South Africa | 101 | 124 | 0.81 || 17 | Tessa Wullaert | Belgium | 101 | 154 | 0.66 || 18 | Tiffeny Milbrett | United States | 100 | 206 | 0.49 |**Key Findings:**- **Total:** 18 female footballers have scored at least 100 international goals at the senior level- **Top Goal Scorer:** Christine Sinclair (Canada) holds the all-time record with 190 goals- **Highest Goals-Per-Match Ratio:** Julie Fleeting (Scotland) and Elisabetta Vignotto (Italy) both have 0.98 goals per match- **Most Represented Country:** United States leads with 7 players in the top 18- **Active Players:** Players in bold are still active at international levelThis data directly addresses the user's question about finding women's soccer players with 100+ international goals and will help determine who has the highest goals-per-match ratio among them.<|end_search_result|>\n\nAssistant thinks: Based on the search results, I have found the complete list of women's soccer players with 100 or more international goals. Now I need to identify who has the highest goals-per-match ratio among them.\n\nLooking at the data:\n\n| Player | Nation | Goals | Caps | Goals per Match |\n|--------|--------|-------|------|-----------------|\n| Christine Sinclair | Canada | 190 | 331 | 0.57 |\n| Abby Wambach | United States | 184 | 255 | 0.72 |\n| Mia Hamm | United States | 158 | 276 | 0.57 |\n| Maysa Jbarah | Jordan | 134 | 140 | 0.96 |\n| Carli Lloyd | United States | 134 | 316 | 0.42 |\n| Kristine Lilly | United States | 130 | 354 | 0.37 |\n| Birgit Prinz | Germany | 128 | 214 | 0.60 |\n| Alex Morgan | United States | 123 | 224 | 0.55 |\n| Marta | Brazil | 121 | 200 | 0.61 |\n| Julie Fleeting | Scotland | 116 | 118 | 0.98 |\n| Patrizia Panico | Italy | 110 | 204 | 0.54 |\n| Elisabetta Vignotto | Italy | 107 | 109 | 0.98 |\n| Michelle Akers | United States | 107 | 155 | 0.69 |\n| Carolina Morace | Italy | 105 | 153 | 0.69 |\n| Vivianne Miedema | Netherlands | 104 | 132 | 0.79 |\n| Portia Modise | South Africa | 101 | 124 | 0.81 |\n| Tessa Wullaert | Belgium | 101 | 154 | 0.66 |\n| Tiffeny Milbrett | United States | 100 | 206 | 0.49 |\n\nThe highest goals-per-match ratio is 0.98, shared by:\n1. **Julie Fleeting** (Scotland) - 116 goals in 118 caps\n2. **Elisabetta Vignotto** (Italy) - 107 goals in 109 caps\n\nBoth players have the same goals-per-match ratio of 0.98. Julie Fleeting has slightly more goals (116 vs 107) and slightly more caps (118 vs 109), but their ratios are identical.\n\nSince both players share the highest ratio, I should note that Julie Fleeting and Elisabetta Vignotto are tied for the highest goals-per-match ratio among women's soccer players with 100+ international goals.\n\n\\boxed{Julie Fleeting and Elisabetta Vignotto (tied at 0.98 goals per match)}",
    "Pred_Answer": "Julie Fleeting and Elisabetta Vignotto (tied at 0.98 goals per match)"
  },
  {
    "id": 37,
    "Question": "<|im_start|>user\nYou are a reasoning assistant with the ability to perform web searches to help you answer the user's question accurately. You have special tools:\n\n- To perform a search: write <|begin_search_query|> your query here <|end_search_query|>.\nThen, the system will search and analyze relevant web pages, then provide you with helpful information in the format <|begin_search_result|> ...search results... <|end_search_result|>.\n\nYou can repeat the search process multiple times if necessary. The maximum number of search attempts is limited to 10.\n\nOnce you have all the information you need, continue your reasoning.\n\nExample:\nQuestion: \"Alice David is the voice of Lara Croft in a video game developed by which company?\"\nAssistant thinking steps:\n- I need to find out who voices Lara Croft in the video game.\n- Then, I need to determine which company developed that video game.\n\nAssistant:\n<|begin_search_query|>Alice David Lara Croft voice<|end_search_query|>\n\n(System returns processed information from relevant web pages)\n\nAssistant thinks: The search results indicate that Alice David is the voice of Lara Croft in a specific video game. Now, I need to find out which company developed that game.\n\nAssistant:\n<|begin_search_query|>video game developed by Alice David Lara Croft<|end_search_query|>\n\n(System returns processed information from relevant web pages)\n\nAssistant continues reasoning with the new information...\n\nRemember:\n- Use <|begin_search_query|> to request a web search and end with <|end_search_query|>.\n- When done searching, continue your reasoning.\n\nPlease answer the following question. You should think step by step to solve it.\n\nProvide your final answer in the format \\boxed{YOUR_ANSWER}.\n\nQuestion:\nWho is the only female artist to have topped Spotify's list of most-streamed artists in three consecutive years (2020–2022)?<|im_end|>\n<|im_start|>assistant\n<think>\n",
    "answer": "No one",
    "Output": "The user is asking about a female artist who topped Spotify's list of most-streamed artists for three consecutive years: 2020, 2021, and 2022.\n\nI need to search for information about Spotify's most-streamed artists list for these specific years.\n\nLet me start by searching for this information.\n</think>\n\n<|begin_search_query|>Spotify most-streamed artists 2020 2021 2022 female artist<|end_search_query|>\n\n<|begin_search_result|>Based on the analyzed web pages, the information regarding the most-streamed artists on Spotify for the years 2020, 2021, and 2022 reveals the following:- **Global Top Artists:** According to Web Page 1 (Spotify Newsroom, Dec 2025), **Bad Bunny** was the Global Top Artist for the fourth consecutive time, having claimed the title in **2020, 2021, and 2022**.- **Top Female Artists:** Web Page 6 (Threads, Dec 2025) explicitly lists the most-streamed female artists on Spotify each year according to Spotify Wrapped:  - **2020:** Billie Eilish  - **2021:** Taylor Swift  - **2022:** Taylor Swift- **Verification:** Web Page 7 (Chart Data, Dec 2021) confirms that Taylor Swift ranked as the top female artist in 2021 globally.**Conclusion:** There is no single female artist who topped Spotify's list of most-streamed artists for three consecutive years covering 2020, 2021, and 2022. Instead, **Billie Eilish** topped the list in 2020, while **Taylor Swift** topped the list for female artists in both 2021 and 2022. Additionally, the overall top artist for all three years (2020-2022) was male (Bad Bunny).<|end_search_result|>\n\nAssistant thinks: The search results indicate that there is no female artist who topped Spotify's list of most-streamed artists for three consecutive years (2020-2022). According to the information:\n- **2020:** Billie Eilish was the top female artist\n- **2021:** Taylor Swift was the top female artist\n- **2022:** Taylor Swift was the top female artist\n\nSo Taylor Swift topped the list for female artists in 2021 and 2022 (two consecutive years), but not 2020. Billie Eilish topped in 2020.\n\nLet me search more specifically to verify if there's any female artist who topped the overall most-streamed artists list (not just female category) for three consecutive years.\n</think>\n\n<|begin_search_query|>Spotify Wrapped 2020 2021 2022 top artists list female<|end_search_query|>\n\n<|begin_search_result|>Based on the analyzed web pages, here is the comprehensive information regarding Spotify Wrapped top artists for 2020, 2021, and 2022:**Overall Most-Streamed Artists (Global):**- **2020:** Bad Bunny (male)- **2021:** Bad Bunny (male)- **2022:** Bad Bunny (male)According to Wikipedia (Web Page 2) and Variety (Web Page 7), Bad Bunny is the only artist to have topped the overall Spotify most-streamed artists list for three consecutive years (2020-2022).**Top Female Artists:**- **2020:** Billie Eilish (top female artist)- **2021:** Taylor Swift (top female artist)- **2022:** Taylor Swift (top female artist)According to Threads (Web Page 6) and multiple sources, Taylor Swift topped the female artist list in 2021 and 2022, but Billie Eilish topped the female artist list in 2020.**Important Clarification:**There is **no female artist** who topped Spotify's overall most-streamed artists list for three consecutive years covering 2020, 2021, and 2022. The premise of the question appears to be incorrect. **Additional Context:**- According to Wikipedia (Web Page 2), Taylor Swift is the **only female artist** to have topped the overall Spotify most-streamed artists list (2023-2024), not during the 2020-2022 period.- Bad Bunny (male) is the only artist to have achieved three consecutive years at the top of the overall list (2020-2022).- In 2022, Taylor Swift was ranked #2 globally and was the most-streamed female artist of that year (Web Page 5, Web Page 7, Web Page 9).**Summary:** The confusion likely stems from mixing up the overall artist ranking with the female artist ranking. No female artist has ever topped the overall Spotify most-streamed artists list for three consecutive years.<|end_search_result|>\n\nAssistant thinks: Based on the search results, I now have clear information:\n\n1. **Overall Most-Streamed Artists (2020-2022):** Bad Bunny (male) topped the list for all three years\n2. **Top Female Artists (2020-2022):**\n   - 2020: Billie Eilish\n   - 2021: Taylor Swift\n   - 2022: Taylor Swift\n\nSo Taylor Swift topped the female artist list for 2021 and 2022 (two consecutive years), but not 2020.\n\nHowever, the question asks about topping Spotify's list of \"most-streamed artists\" which could refer to the overall list, not specifically the female category. Let me search more specifically to see if there's any clarification about whether the question refers to the overall list or female-specific list.\n\nActually, based on all the search results, the answer appears to be that **there is no female artist** who topped Spotify's overall most-streamed artists list for three consecutive years (2020-2022). The question premise may be incorrect.\n\nLet me do one more search to verify this information more precisely.\n</think>\n\n<|begin_search_query|>Spotify most-streamed artists 2020 2021 2022 who topped the list<|end_search_query|>\n\n<|begin_search_result|>Based on the analyzed web pages, the following information confirms the most-streamed artists on Spotify for the years 2020, 2021, and 2022:- **2020:** **Bad Bunny** topped the list (approx. 8.3 billion streams).- **2021:** **Bad Bunny** topped the list (approx. 9.1 billion streams).- **2022:** **Bad Bunny** topped the list (approx. 18.5 billion streams).**Key Findings:**- **Consecutive Record:** According to Wikipedia and Spotify's official newsroom, **Bad Bunny** is the only artist to have topped the overall Spotify most-streamed artists list for three consecutive years (2020–2022).- **Transition to Taylor Swift:** **Taylor Swift** took over the #1 spot starting in **2023** and held it through 2024.- **Female Artist Context:** While Bad Bunny dominated the overall list for 2020–2022, **Billie Eilish** was the top female artist in 2020, and **Taylor Swift** was the top female artist in 2021 and 2022. No female artist has topped the *overall* most-streamed list for three consecutive years during this specific period.<|end_search_result|>\n\nAssistant thinks: Based on all the search results, I now have a clear picture:\n\n1. **Overall Most-Streamed Artists (2020-2022):** Bad Bunny topped the list for all three years\n2. **Top Female Artists (2020-2022):** Billie Eilish (2020), Taylor Swift (2021), Taylor Swift (2022)\n\nThe question asks about a \"female artist\" who topped Spotify's list of most-streamed artists for three consecutive years (2020-2022). Based on the search results:\n\n- **Bad Bunny** (male) is the only artist to have topped the overall list for three consecutive years (2020-2022)\n- **Taylor Swift** topped the female artist list in 2021 and 2022 (two consecutive years), but not 2020\n- **Billie Eilish** topped the female artist list in 2020\n\nThere appears to be **no female artist** who topped Spotify's overall most-streamed artists list for three consecutive years covering 2020-2022. The question's premise may be incorrect, or it could be referring to the female-specific category where Taylor Swift topped for 2021-2022 (but not 2020).\n\nHowever, if the question is asking about the overall most-streamed artists list, the answer would be that **no female artist** achieved this feat during 2020-2022. Bad Bunny (male) was the only artist to achieve three consecutive years at the top.\n\nLet me provide the answer based on the most accurate interpretation of the question.\n</think>\n\nBased on my research, I can now provide a clear answer to this question.\n\n**Step-by-Step Analysis:**\n\n1. **Spotify's Most-Streamed Artists (2020-2022):**\n   - **2020:** Bad Bunny topped the list\n   - **2021:** Bad Bunny topped the list\n   - **2022:** Bad Bunny topped the list\n\n2. **Top Female Artists During This Period:**\n   - **2020:** Billie Eilish\n   - **2021:** Taylor Swift\n   - **2022:** Taylor Swift\n\n3. **Key Findings:**\n   - **Bad Bunny** (male) is the only artist to have topped Spotify's overall most-streamed artists list for three consecutive years (2020-2022)\n   - **Taylor Swift** is the only female artist to have topped the overall list (but starting in 2023, not during 2020-2022)\n   - No female artist topped the overall most-streamed list for three consecutive years during the 2020-2022 period\n\n**Conclusion:**\n\nThe question's premise appears to be incorrect. There is **no female artist** who topped Spotify's overall most-streamed artists list for three consecutive years (2020-2022). **Bad Bunny** (male) achieved this feat during those years.\n\nHowever, if the question refers to the **female artist category specifically**, then **Taylor Swift** topped the female artist list for 2021 and 2022 (two consecutive years), but not 2020 (when Billie Eilish was #1).\n\n\\boxed{No female artist topped Spotify's overall most-streamed artists list for three consecutive years (2020-2022); Bad Bunny (male) was the only artist to achieve this during that period.}",
    "Pred_Answer": "No female artist topped Spotify's overall most-streamed artists list for three consecutive years (2020-2022); Bad Bunny (male) was the only artist to achieve this during that period."
  }









# xbench

In [ ]:
import json
import re

# ========== 配置路径 ==========
input_path  = "/home/shaolingxuan/project/Search-o1/outputs/runs.baselines/XbenchDS.qwen3.5-9b.search_o1/test.8.9,22:3-中文.json"
output_path = "/home/shaolingxuan/project/Search-o1/outputs/runs.baselines/XbenchDS.qwen3.5-9b.search_o1/all-clean.json"

# ========== 1. 读取已有的 all_extracted.json ==========
try:
    with open(output_path, "r", encoding="utf-8") as f:
        all_data = json.load(f)
    print(f"读取已有数据 {len(all_data)} 条")
except (FileNotFoundError, json.JSONDecodeError):
    all_data = []
    print("输出文件不存在或为空，从头开始")

# ========== 2. 读取输入 JSON 文件 ==========
with open(input_path, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

# 如果是单条 dict，包装成 list
if isinstance(raw_data, dict):
    raw_data = [raw_data]

print(f"输入文件共 {len(raw_data)} 条数据")

# ========== 3. 正则匹配 \boxed{...} ==========
pattern = r'\\boxed\{(.*)\}'    # ← 用非贪婪 .*?，见下方说明。保持和webthinker一致，就用贪婪模式

matched_count   = 0
unmatched_count = 0

for item in raw_data:
    # 兼容 "Output" 和 "output" 两种字段名
    output = item.get("Output", item.get("output"))
    if output is None:
        print(f"  [跳过] id={item.get('id')} 无 Output 字段")
        unmatched_count += 1
        continue

    matches = re.findall(pattern, output)
    if matches:
        pred_answer = matches[-1]  # 取最后一个匹配
        extracted = {
            "id":           item.get("id"),
            "Question":     item.get("Question"),
            "answer":       item.get("answer"),
            "Output":       output,
            "Pred_Answer":  pred_answer
        }
        all_data.append(extracted)
        matched_count += 1
        print(f"  [匹配] id={item.get('id')}  Pred_Answer={pred_answer}")
    else:
        unmatched_count += 1
        print(f"  [未匹配] id={item.get('id')}  Output 中无 \\boxed{{}}")

# ========== 4. 写回 all_extracted.json ==========
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_data, f, ensure_ascii=False, indent=2)

print(f"\n完成！匹配成功 {matched_count} 条，未匹配 {unmatched_count} 条")
print(f"输出文件现有 {len(all_data)} 条数据，路径: {output_path}")


读取已有数据 43 条


FileNotFoundError: [Errno 2] No such file or directory: '/home/shaolingxuan/project/Search-o1/outputs/runs.baselines/XbenchDS.qwen3.5-9b.search_o1/dev.8.10,13:9.96.json'